# Exploracao dos Parquets no Google Colab

Este caderno e para olhar os Parquets com calma: escolher uma base, carregar um `DataFrame`, ver `head`, `describe`, contagens basicas e abrir o campo `texto` completo sem truncamento.

Fluxo recomendado: escolha a base, rode a carga, ajuste filtros, use a tabela compacta para encontrar um registro e abra o texto integral por `texto_id` ou indice.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/pedblan/falando_nela.git'
REPO_DIR = Path('/content/falando_nela')
REPO_REF = ''  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags', '--prune'], check=True)
    if not REPO_REF:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

if REPO_REF:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Repo em:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
PARQUET_ROOT = DATA_ROOT / 'processed' / 'textos_parlamentares' / 'v1' / 'parquet'
os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)

SELECTED_PARQUET = None     # exemplo: 'camara__plenario_discursos.parquet'
MAX_ROWS = None             # use um inteiro para uma leitura parcial exploratoria
LOAD_TEXT_COLUMN = True     # mantenha True para abrir texto integral
COMPACT_TABLE_ROWS = 100
VALUE_COUNT_TOP_N = 20

print('PARQUET_ROOT:', PARQUET_ROOT)


In [ ]:
import html
import io
import re
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import HTML, Markdown, display

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 180)

KEY_COLUMNS = [
    'texto_id',
    'source',
    'dataset',
    'casa',
    'ambito',
    'orgao_sigla',
    'documento_tipo',
    'unidade_analitica',
    'data',
    'ano',
    'mes',
    'titulo',
    'parlamentar_nome',
    'parlamentar_partido',
    'parlamentar_uf',
    'proposicao_identificacao',
    'documento_classe',
    'status_deliberativo',
    'texto_tamanho',
    'texto_status',
    'url_texto',
    'raw_path',
]
VALUE_COUNT_COLUMNS = [
    'source',
    'dataset',
    'documento_tipo',
    'unidade_analitica',
    'ano',
    'mes',
    'ambito',
    'orgao_sigla',
    'parlamentar_partido',
    'parlamentar_uf',
    'documento_classe',
    'status_deliberativo',
    'texto_status',
]
TEXT_METADATA_COLUMNS = [
    'texto_id',
    'source',
    'dataset',
    'data',
    'documento_tipo',
    'unidade_analitica',
    'titulo',
    'parlamentar_nome',
    'proposicao_identificacao',
    'orgao_sigla',
    'texto_tamanho',
    'url_texto',
    'raw_path',
]


def require_parquet_root(path: Path) -> list[Path]:
    if not path.exists():
        raise FileNotFoundError(f'Diretorio de Parquets nao encontrado: {path}')
    files = sorted(path.glob('*.parquet'))
    if not files:
        raise FileNotFoundError(f'Nenhum arquivo .parquet encontrado em: {path}')
    return files


def choose_selected_path(parquet_files: list[Path], selected_name: str | None) -> Path:
    names = [path.name for path in parquet_files]
    if selected_name is None:
        selected_name = names[0]
    if selected_name not in names:
        raise ValueError(f'SELECTED_PARQUET={selected_name!r} nao esta em {names}')
    return next(path for path in parquet_files if path.name == selected_name)


def load_parquet_frame(path: Path, *, max_rows: int | None, load_text_column: bool) -> pd.DataFrame:
    schema = pq.read_schema(path)
    columns = list(schema.names)
    if not load_text_column and 'texto' in columns:
        columns = [column for column in columns if column != 'texto']
    if max_rows is None:
        return pd.read_parquet(path, columns=columns)
    parquet_file = pq.ParquetFile(path)
    batches = parquet_file.iter_batches(batch_size=max_rows, columns=columns)
    try:
        batch = next(batches)
    except StopIteration:
        return pd.DataFrame(columns=columns)
    return batch.to_pandas()


def compact_columns(df: pd.DataFrame) -> list[str]:
    return [column for column in KEY_COLUMNS if column in df.columns and column != 'texto']


def display_table(frame: pd.DataFrame, *, rows: int = 100) -> None:
    frame = frame.head(rows).copy()
    try:
        from itables import init_notebook_mode, show
        init_notebook_mode(all_interactive=True)
        show(frame)
    except Exception as exc:
        print(f'Visualizacao interativa indisponivel ({exc}). Usando display pandas.')
        display(frame)


def print_info(df: pd.DataFrame) -> None:
    buffer = io.StringIO()
    df.info(buf=buffer)
    display(HTML('<pre style="white-space: pre-wrap">' + html.escape(buffer.getvalue()) + '</pre>'))


def basic_describe(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return df.describe(include='all').T


def null_report(df: pd.DataFrame) -> pd.DataFrame:
    total = len(df)
    report = pd.DataFrame({
        'nulos': df.isna().sum(),
        'preenchidos': df.notna().sum(),
    })
    report['pct_nulos'] = (report['nulos'] / total * 100).round(2) if total else 0
    return report.sort_values(['nulos', 'preenchidos'], ascending=[False, True])


def value_counts_report(df: pd.DataFrame, *, top_n: int) -> dict[str, pd.DataFrame]:
    reports = {}
    for column in VALUE_COUNT_COLUMNS:
        if column not in df.columns:
            continue
        series = df[column].dropna()
        if series.empty:
            continue
        counts = series.value_counts().head(top_n).rename_axis(column).reset_index(name='registros')
        reports[column] = counts
    return reports


def contains_filter(series: pd.Series, text: str | None) -> pd.Series:
    if not text:
        return pd.Series(True, index=series.index)
    return series.fillna('').astype(str).str.contains(re.escape(text), case=False, na=False)


def apply_filters(
    df: pd.DataFrame,
    *,
    ano: str | None = None,
    mes: str | None = None,
    documento_tipo: str | None = None,
    parlamentar: str | None = None,
    proposicao: str | None = None,
    orgao_sigla: str | None = None,
    busca_textual: str | None = None,
) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    equality_filters = {
        'ano': ano,
        'mes': mes,
        'documento_tipo': documento_tipo,
        'orgao_sigla': orgao_sigla,
    }
    for column, value in equality_filters.items():
        if value and column in df.columns:
            mask &= df[column].fillna('').astype(str).eq(str(value))
    if parlamentar and 'parlamentar_nome' in df.columns:
        mask &= contains_filter(df['parlamentar_nome'], parlamentar)
    if proposicao and 'proposicao_identificacao' in df.columns:
        mask &= contains_filter(df['proposicao_identificacao'], proposicao)
    if busca_textual and 'texto' in df.columns:
        mask &= contains_filter(df['texto'], busca_textual)
    return df.loc[mask].copy()


def render_full_text(row: pd.Series) -> None:
    metadata = {column: row.get(column) for column in TEXT_METADATA_COLUMNS if column in row.index and pd.notna(row.get(column))}
    display(Markdown('### Metadados do texto selecionado'))
    display(pd.DataFrame([metadata]).T.rename(columns={0: 'valor'}))
    text = row.get('texto') if 'texto' in row.index else None
    if not isinstance(text, str) or not text:
        display(Markdown('**Este DataFrame nao tem a coluna `texto` carregada. Volte a `LOAD_TEXT_COLUMN = True` e recarregue.**'))
        return
    escaped_text = html.escape(text)
    display(Markdown('### Texto integral'))
    display(HTML(
        '<pre style="white-space: pre-wrap; font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; '
        'font-size: 14px; line-height: 1.45; border: 1px solid #ddd; padding: 16px; border-radius: 6px; '
        'max-height: none; overflow: visible;">' + escaped_text + '</pre>'
    ))


## Escolher a base

A lista abaixo mostra os Parquets disponiveis. Altere `SELECTED_PARQUET` na celula de parametros ou use o seletor quando ele aparecer, antes de carregar o `DataFrame`.


In [ ]:
parquet_files = require_parquet_root(PARQUET_ROOT)
print('Parquets disponiveis:')
for index, path in enumerate(parquet_files):
    print(f'{index}: {path.name}')

try:
    import ipywidgets as widgets
    parquet_selector = widgets.Dropdown(
        options=[path.name for path in parquet_files],
        value=SELECTED_PARQUET or parquet_files[0].name,
        description='Base:',
        layout=widgets.Layout(width='650px'),
    )
    display(parquet_selector)
    print('Depois de escolher no seletor, rode a proxima celula.')
except Exception as exc:
    parquet_selector = None
    print(f'Seletor interativo indisponivel ({exc}). Use SELECTED_PARQUET manualmente.')


## Carregar DataFrame

Esta celula carrega a base selecionada. Para bases muito grandes, ajuste `MAX_ROWS`. Para inspecionar texto integral, mantenha `LOAD_TEXT_COLUMN = True`.


In [ ]:
selected_name = SELECTED_PARQUET
if selected_name is None and 'parquet_selector' in globals() and parquet_selector is not None:
    selected_name = parquet_selector.value

selected_path = choose_selected_path(parquet_files, selected_name)
df = load_parquet_frame(selected_path, max_rows=MAX_ROWS, load_text_column=LOAD_TEXT_COLUMN)

print('Arquivo selecionado:', selected_path)
print('Linhas e colunas:', df.shape)
print('Coluna texto carregada:', 'texto' in df.columns)
print('Colunas:')
print(list(df.columns))


## Primeira olhada

`shape` mostra o tamanho da tabela. `head` mostra as primeiras linhas. A tabela compacta remove `texto` para facilitar navegação.


In [ ]:
print('df.shape =', df.shape)
display(df.head())

compact_df = df[compact_columns(df)].copy() if compact_columns(df) else df.drop(columns=['texto'], errors='ignore').copy()
display_table(compact_df, rows=COMPACT_TABLE_ROWS)


## Schema, describe e nulos

`info` mostra tipos e memoria. `describe(include="all")` resume colunas numericas e categoricas. O relatorio de nulos mostra preenchimento por campo.


In [ ]:
print_info(df)

describe_df = basic_describe(df.drop(columns=['texto'], errors='ignore'))
display(describe_df)

nulos_df = null_report(df)
display(nulos_df)


## Value counts basicos

Contagens rapidas para campos categoricos centrais. Ajuste `VALUE_COUNT_TOP_N` para ver mais ou menos categorias.


In [ ]:
counts = value_counts_report(df, top_n=VALUE_COUNT_TOP_N)
for column, table in counts.items():
    print(f'\n{column}')
    display(table)


## Filtros simples

Edite os valores abaixo e rode a celula. Use filtros para reduzir a tabela antes de abrir textos completos.


In [ ]:
FILTER_ANO = None                  # exemplo: '2026'
FILTER_MES = None                  # exemplo: '05'
FILTER_DOCUMENTO_TIPO = None       # exemplo: 'discurso'
FILTER_PARLAMENTAR = None          # trecho do nome
FILTER_PROPOSICAO = None           # trecho como 'PEC 38/2011'
FILTER_ORGAO_SIGLA = None          # exemplo: 'CCJ'
SEARCH_TEXT = None                 # palavra ou expressao dentro de texto

filtered_df = apply_filters(
    df,
    ano=FILTER_ANO,
    mes=FILTER_MES,
    documento_tipo=FILTER_DOCUMENTO_TIPO,
    parlamentar=FILTER_PARLAMENTAR,
    proposicao=FILTER_PROPOSICAO,
    orgao_sigla=FILTER_ORGAO_SIGLA,
    busca_textual=SEARCH_TEXT,
)

print('Registros filtrados:', filtered_df.shape)
filtered_compact = filtered_df[compact_columns(filtered_df)].copy() if compact_columns(filtered_df) else filtered_df.drop(columns=['texto'], errors='ignore').copy()
display_table(filtered_compact, rows=COMPACT_TABLE_ROWS)


## Escolher um texto integral

Use a tabela filtrada para copiar um `texto_id`, ou escolha um indice. A celula imprime o campo `texto` completo, sem truncamento.


In [ ]:
TEXT_ID = None       # cole um texto_id aqui para abrir um registro especifico
ROW_INDEX = 0        # usado quando TEXT_ID fica vazio; considera a tabela filtrada

source_df = filtered_df if 'filtered_df' in globals() and not filtered_df.empty else df
if TEXT_ID:
    matches = df[df['texto_id'].astype(str).eq(str(TEXT_ID))] if 'texto_id' in df.columns else pd.DataFrame()
    if matches.empty:
        raise ValueError(f'texto_id nao encontrado: {TEXT_ID}')
    selected_row = matches.iloc[0]
else:
    if source_df.empty:
        raise ValueError('Nao ha registros para selecionar.')
    selected_row = source_df.iloc[ROW_INDEX]

render_full_text(selected_row)
